# Benchmark

**A Stochastic Programming Model for Dynamic Portfolio Management with Financial Derivatives**

*D. Barro, G. Consigli, V. Varun. A stochastic programming model for dynamic portfolio management with financial derivatives. Journal of Banking and Finance, Volume 140, 2022. https://doi.org/10.1016/j.jbankfin.2022.106445*

___
___

*The paper builds an arbitrage-free scenario tree by bootstrapping historical joint returns of the relevant assets, preserving their empirical correlations. Derivatives are then priced on that tree via a minimal-entropy risk-neutral measure and backward induction. With all prices fixed, a linear program solves for the trading policy (asset and derivative positions at each node) that maximizes a tunable trade-off between expected wealth and tail risk, subject to wealth and position constraints. This is re-solved on a rolling basis as new data arrives, so the policy adapts to the current regime.*

*Empirically, results are strong; its strengths are an exact, interpretable optimum with hard constraints enforced, and regime-adaptive sizing; its main weaknesses are that the policy is only optimal for the specific tree solved, is sensitive to how well bootstrapping represents future markets, scales poorly with dimension, and must be re-solved from scratch each period rather than yielding a reusable policy.*

___

### Summary

1. Build the tree
    - Each node knows 4 stochastic values (equity, bond, money market, VIX) + a risk free rate (deterministic) used separately.
    - The 4 stochastic assets get bootstrapped to build the tree: for each node, a children is created by sampling a historical return vectors of these assets: to keep there historical correlation. 
    - Each node must have at least 4 children to solve non-arbitrage conditions. The paper uses monthly nodes for 6 months, and the first node has 10 childen, that is, total number of nodes is $\small 1 + 10 + 10 \ (4) + 10 \ (4^2) + 10 \ (4^3) + ...$
    - Then, we have a full arbitrage-free tree of asset prices under the physical measure.


2. Price derivatives on that tree
    - Once we have the full arbitrage-free tree of asset prices with physical measure $p(n)$, fit the minimal-entropy risk-neutral measure $q(n)$ node-by-node (same tree, different probabilities), then get the option prices at every node by backward induction under risk-neutral $q$

3. Solve the portfolio LP
    - By this point, every price on every node is a fixed number, computed and done. The only remaining unknowns are the trading decisions ($x_{in}$, $c^h_{1n}(j,k)$, etc.). Solve

        $$\max_{x,c,p} \,\,\,\, (1-\lambda) \ \mathbb E[W_{N_T}] - \lambda \ \text{CVaR}_{\alpha}(W_{N_T})$$

        using the physical probabilities $p(n)$, subject to the wealth / cash / inventory-balance /position-bound constraints, plus whichever strategy-shape constraints you're imposing if any (protective put / covered call / straddle, ...).

4. Rolling / Receding-horizon re-optimization
    - At $t=0$, run steps 1,2,3, get the decision and execute it.
    - Let one period pass, now at $t=1$ observe the actual realized market return
    - At this new date, rebuild a fresh tree from stage 1 (using updated historical data up to now), re-price derivative (stage 2), re-solve LP (stage 3), then get a new decision and execute it
    - Repeat

5. Evaluation (Out-of-sample check)
    - After solving, find whichever path is closest (Euclidean distance) to what market actually did, and read off that path's node-level decisions to mark-to-market the strategy at realized prices. Roll forward to test on longer period, using period final wealth as the next period initial wealth.
    - This checks whether the built tree was good or not (even if the policy is optimal on that tree, the tree might be a misrepresentation of the market)

**Advantages**
- Reshape historical return distribution
- Empirically strong results
- Regime-adaptive sizing, not a fixed rule
    - Optimizer decides when, how much and which strikes/maturities
- Tunable risk appetite
    - $\lambda$ gives a single and interpretable choice between growth and risk
- Fully interpretable
- Exact optimum from stated objective
    - LP solver finds optimal policy
- Hard constraints are exactly enforced 

**Disadvantages**
- Can only trade options with expiry within the model
- Curse of dimensionality
- Policy is optimal for that one finite tree
    - The tree might be a misrepresentation of the market
- Historical bootstrap dependence
- No transferable policy function
    - Every rebalancing require new solving, cannot learn a policy and evaluate instantly like a trained neural policy
- Option pricing under incompleteness is a modeling choice
    - MEM/entropy measure is one defensible way to pick among the infinitely many risk-neutral measures consistent with no-arbitrage in an incomplete market.

___
### Section 2: Optimal investment model with options

**2.1 Definition and Variables**

Discrete market: information evolve according to a non recombining scenario tree. The nodes in the tree $n \in \mathcal{N}_t$ at stage t, where every $n$ has a unique parent $n- \in \mathcal{N}_{t-1}$ and children $n+ \in \mathcal{N}_{t+1}$ with cardinality $\# n+ $ for $t \leq T-1$. For every $n \in \mathcal{N}_T$, a scenario is a path $\omega_n, \omega_{n-}, \omega_{n--}, ..., \omega_{n_0}$, with $n_0$ the root node.

Time: consider a discrete and finite decision horizon $t \in \mathcal{T}$, $\mathcal{T} := {t_0, t_1, ..., t_J = T}$. In this case, use a 6-month planning horizon $T=6$ with monthly steps.

Tree with time: For each node $n \in \mathcal{N}_t$, denote $t_n$ its associated time.

Assets: Consider the asset $i \in \mathcal{I}$, which include equity (S&P500), bond, money market indices, volatility index (VIX) and equity options.

Random parameters:

- $r_n$ is the risk-free rate in node $n$ for the period from $t_n$ to $t_{n+}$.
- $v_{in}$ is the price of asset $i$, in node $n$ and we indicate with $\rho_{i,n} = \frac{v_{in}}{v_{in_-}} - 1$ the asset return over the period $t_{n-} - t_n$.
- $O^c_{1n}(j,k)$ is the price in node $n$ of a European call option on the asset $i=1$ (equity) with strike price equal to $k$ and expiry $t_j$.
- $O^p_{1n}(j,k)$ is the price in node $n$ of a European put option on asset $1$ with strike $k$ and expiry $t_j$.

Decision variables:

- $x^0_n$ is the cash balance at node $n$,
- $x^+_{in}$ nominal amount of asset $i$ purchased in node $n$,
- $x^-_{in}$ nominal amount of asset $i$ sold in node $n$,
- $x_{in}$ amount of asset $i$ held in node $n$,
- $\mathcal{C}^{h*}_{1n}(j,k)$ define a generic option position on the equity with maturity $t_j$ and strike $k$ in node $n$, where $\mathcal{C} = \{c, p\}$ (call, put), $h = \{l, s\}$ (long, short), $*=\{\emptyset, +, -\}$ (current position, purchases, sales).

**2.2 Model Setup**

Optimization problem: define simply investments, call, put by $x,c,p$, the decision variables to optimize. The objective function is based on the terminal wealth and is defined through a canonical risk-reward function, whose trade-off is determined by a coefficient $\lambda \in [0,1]$ and, whose risk measure is the terminal CVaR with tolerance $\alpha$.
$$
\max_{x,c,p} \,\,\,\, (1-\lambda) \cdot \mathbb{E}[W_{n \in \mathcal{N}_T}] - \lambda \cdot \text{CVaR}_{\alpha}(W_{n \in \mathcal{N}_T}) \tag{1}
$$

Conditions: for all $t \in \mathcal{T}$, $n \in \mathcal{N}_t$ and given initial conditions $x^0_0, x_{i,0}, W_0$ (no options in the initial portfolio):

Wealth equation:
$$W_n = x^0_n + \sum_{i \in \mathcal{I}} x_{in}\, v_{1n} + NCP_n + NPP_n \tag{2}$$
- At each node $n$, this is the sum of the cash surplus, value of investment plus long minus short positions.
- $NCP$ and $NPP$ stands for net call/put position and are defined as $\small NCP_n = \sum_{t_n < t_j \le T-1,\,k} \left[ c^{l}_{1n}(j,k)\, O^{c}_{1n}(j,k) - c^{s}_{1n}(j,k)\, O^{c}_{1n}(j,k) \right] \,\, (8,9)$. Change $c$ for $p$ in $NPP$. Note this excludes $t=T$.

Cash balance constraint: (similar to self-financing portfolio)
$$x^0_n = x^0_{n_-}\left(1 + r_{n_-}\right) + \sum_{i \in \mathcal{I}} x^-_{in}\, v_{1n}\, \delta^- - \sum_{i \in \mathcal{I}} x^+_{in}\, v_{1n}\, \delta^+ + C_n + P_n \tag{3}$$
- depends on cash available at beginning of period and interests, on asset selling and buying and on options.
- $C_n = C_n^l + C_n^s$ and $P_n = P_n^l + P_n^s$, for all $n, t$, are the cash coming in from options trading and cash settlements at expiry. $(10, 11)$
- Before expiry, option position can be traded at current price. At expiry, consider cash settlements.

Asset inventory balance:
$$x_{i,n} = x_{i,n_-} + x^+_{i,n} - x^-_{i,n} \quad \forall i \in \mathcal{I} \tag{4}$$
- standard stage by stage rebalancing decisions on assets.

Option contracts inventory balance:
$$C^h_{1n}(j,k) = C^h_{1n_-}(j,k) + C^{h+}_{1n}(j,k) - C^{h-}_{1n}(j,k) \quad \forall h, j, k \tag{5}$$
- standard stage by stage rebalancing decisions on options.

Bounds for asset proportion in current portfolio:
$$L_i W_n \le x_{in} \le U_i W_n, \quad \forall i \in \mathcal{I} \tag{6}$$
- Asset specific bounds on positions relative to current portfolio

Rule-out some rebalancing decisions at maturity $T$:
$$x^+_{i,n} = x^-_{i,n} = C^{h,+}_{1,n}(j,k) = C^{h,-}_{1,n}(j,k) = C^h_{1n}(j,k) = 0 \quad \\\\ \forall i \in \mathcal{I},\ h \in \mathcal{H},\ n \in \mathcal{N}_T \tag{7}$$
- No rebalancing nor variation in options exposures at end of horizon.


___
### Section 3: Derivatives strategies

Specific additional constraints are applied to the option contracts balance constraints (5) $\small C^h_{1n}(j,k) = C^h_{1n_-}(j,k) + C^{h+}_{1n}(j,k) - C^{h-}_{1n}(j,k) \quad \forall h, j, k$, based on the desired following derivative strategy.


**3.1 Protective put**

**3.2 Covered call**

**3.3 Straddle**

**3.4 Strip and strap**

**3.5 Derivatives-based strategy summary**

___
### Section 4: Tree processes and pricing

Consider a tree process. The tree structure adopted for pricing an option is the same as the one considered to formulate and solve an optimization problem. The associated scenario probabilities need not be the same.

By the fundamental theorem of asset pricing, absence of arbitrage requires the existence of at least one probability measure under which the discounted underlying stock price is a martingale, thus constant expectation.

In this discrete tree-based framework, we rely one two-steps approach: first verify no-arbitrage conditions then determine option price. This article take a simple data-driven method to generate price and return scenarios, and do not require market to be complete.

**4.1 Data-driven arbitrage-free scenario generation**

Generate a vector of joint return by historical bootstrapping. The the scenario tree is tested for absence of arbitrage (Klassen 2002). If the requirement is satisfied, the risk neutral probabilities are computed (Frittelli 2000).

Recall the notation of node $n \in \mathcal{N}_t, t < T$ and $\#n+$ defined the cardinality of nodes $n$'s children. $q_{n,n+}$ and $p_{n,n+}$ are the conditional risk neutral and empirical probabilities.

We use $R_n = 1 + r_n$ the risk-free compounded return in node $n$, and $\rho_{i,n+} = \frac{v_{i,n+}}{v_{i,n}} - 1$ for the one-period return of asset $i \in \mathcal{I}$. According to the asset pricing rule, for every sub-tree and parent node, the asset's expected return must be equal to the risk free interest rate $R_n$.


**4.2 Risk-neutral pricing**

Since the market is incomplete, there won't martingale measure. This article will consider the minimal entropy martingale (MEM) (Frittelli 2000), which is determined by minimizing the relative entropy of the risk neutral measure with respect to the physical measure, which is a desirable property in a derivatives' portfolio problem.

In a finite state incomplete but arbitrage-free with a random risk-free return, the MEM measure can be found by solving the following optimization problem, for $t \in \mathcal{T}$ and $t < T$, on all (one-stage-deep) subtrees, with parent node $n$:

$$\min_{q \in \mathbb{R}^{\#n_+},\, q > 0} \left( \sum_{n_+=1}^{\#n_+} q_{n,n_+} \ln\left(\frac{q_{n,n_+}}{p_{n,n_+}}\right) \right) \tag{16}$$

$$q^T \cdot \mathbf{1} = 1$$
$$q^T \cdot (1 + \rho_{i,n_+}) = R_n \quad i = 1, \ldots, m$$

where for every $n \in \mathcal{N}_t$, $t < T$, $\#n_+$, with $q_{n,n_+}$ and $p_{n,n_+}$ are the conditional risk neutral and empirical probabilities associated with every arc in the subtree originating from $n$. Here, $q^T \cdot (1 + \rho_{i,n_+}) = R_n \quad i = 1, \ldots, m$ force the branch degree $\#n+$ to be equal or greater than the $m$ assets in the investment universe.

Basically, the assets' expected return under the risk neutral probability must be equal to the risk free interest rate $R_n$.


*Algorithm 1: Arbitrage-free scenario tree generation*

Set initial condition:

- Set data history $[t_a, t_b]$ and investment horizon $\mathcal{T} = \{t_0,t_1,...,t_n\}$ with $t_0 \geq t_b$ and $t_n=T$.
- Upload time series $v$ from $[t_a, t_b]$ and compute historical returns $\rho$
- Set $v(0)$ root nodal price at time $t_0=0$

Specify tree structure: For every $t \in \mathcal{T}$, $t<T$ and $\forall n \in \mathcal{N}_t$:
- Set branching degree $\#n+$ for every $t$ and $n \in \mathcal{N}_t$ (symmetric tree)
- Specify Nodal Partion Matrix (NPM) of dimension $\mathcal{N}_T \times \mathcal{T}$ (columns for stages, rows for scenarios)

Generate scenario tree: **while** $t \leq T-1$ and for each node $n \in \mathcal{N}_t$, **do**:
- Randomly generate $\#n+$ uniform random number $u \in [t_a,t_b]$
- Sample, according to $u$, from the matrix of historical return $\tilde{\rho}(u)$
- Specify the NPM-based leaf nodal label
- Compute for every scenario $v(n+) = v(n)(1 + \tilde{\rho}(u)), \forall n+ \in \mathcal{N}_{t+1}$

Check for absence of arbitrage: **while** $t \leq T-1$ and for every $n \in \mathcal{N}_t$, **do**:
- Find the dual variables $\nu_{n+}$ for which $\small \sum_{n+=1}^{\#n+} \nu_{n+}(1+\rho_{i,n+}) = 1, \forall i \in \mathcal{I}$
- (For each asset, at least one child return beats the risk free rate and one underperforms the risk-free rate, or all returns are exactly risk free-rate)
- If there exist a solution $\nu_{n+} > 0$ for $n+=1,...,\#n+$, then there is no arbitrage.
- Else, reject this subtree and generate a new one


*Algorithm 2: Option pricing*


Setup: 
- Set $n_0 = \{1\}$ the root node, $n \in \mathcal{N}_t, 0 \leq t < T$ and for every t: $n+ \in (t+1)$ and branching degree $\#n+$.
- Set $v_{1,n}$ the stock price in every node $n \in \mathcal{N}_t$ for every $t \in \mathcal{T}$
- Set $\rho_{1,n}$ be the corresponding return process and $R_n = 1+r_n$


Probabilities: **while** $t \leq T-1$, for each node $n \in \mathcal{N}_t$, **do**
- Set $p_{n,n+} = \frac{1}{\#n+}$ for the physical condiditional probability
- Solve problem (16) for the conditional risk-neutral probability

Option pricing: 

Calculate the payoff: **while** $t \leq T, t=t_j$ and $n \in \mathcal{N}_{t_j}$, **do**
- Compute the payoff $\mathcal{C}^h_{1n}$ for both calls and puts (no distinction between long and short)

Backward recursion pricing: **while** $t \leq t_{j-1}$ and for each node $n \in \mathcal{N}_t$, **do**
- $\mathcal{C}^h_{1n} = \frac{1}{1+r_n} \ \sum_{n+ \in \mathcal{N}_{t+1}} q_{n,n+} \mathcal{C}_{1n+}^h \,\,\,\, (17)$ 



___
### Section 5: Computation